# LeetCode 1108 — Defanging an IP Address bằng Python

**Link đề:** https://leetcode.com/problems/defanging-an-ip-address/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def defangIPaddr(self, address: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu chuỗi (`str`) trong Python là **bất biến** (immutable) và hệ quả khi "sửa" chuỗi.
2. Thành thạo 4 công cụ xử lý chuỗi nền tảng: duyệt ký tự, `str.replace`, `str.split` + `str.join`, và `re.sub`.
3. Biết vì sao **gom vào list rồi `join`** là cách xây chuỗi chuẩn, còn `+=` trong vòng lặp có thể chậm.
4. Trace từng ký tự, chứng minh tính đúng, phân tích Time/Space cho mỗi cách.
5. Tự giải các biến thể: "refang" (đảo ngược), thay thế tổng quát, kiểm tra IPv4 hợp lệ.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — thay mọi dấu `.` bằng `[.]`

Cho một địa chỉ IPv4 **hợp lệ** `address`. Trả về phiên bản "defanged" (bẻ nanh) của nó: thay **mọi** dấu chấm `"."` bằng `"[.]"`.

| Input | Output |
|---|---|
| `"1.1.1.1"` | `"1[.]1[.]1[.]1"` |
| `"255.100.50.0"` | `"255[.]100[.]50[.]0"` |

**Constraints:** địa chỉ đã cho là IPv4 hợp lệ.

**Bối cảnh:** "defang" dùng trong bảo mật — khi chia sẻ một IP/URL độc hại trong báo cáo, người ta viết `1[.]2[.]3[.]4` để nó **không thể bấm/kết nối nhầm**.

**IPv4 hợp lệ** gồm 4 số từ `0` đến `255`, ngăn cách bởi đúng 3 dấu chấm. Vậy output luôn dài hơn input đúng `3 × 2 = 6` ký tự (mỗi `.` thành 3 ký tự).

In [1]:
address = "255.100.50.0"
expected = "255[.]100[.]50[.]0"
print("số dấu chấm:", address.count("."))
print("độ dài input:", len(address), "| output:", len(expected), "| chênh:", len(expected) - len(address))

số dấu chấm: 3
độ dài input: 12 | output: 18 | chênh: 6


## 2. Nền tảng: chuỗi trong Python

### 2.1. Chuỗi là dãy ký tự, truy cập bằng chỉ số

`s[i]` là ký tự ở vị trí `i` (bắt đầu từ 0). `len(s)` là số ký tự. Duyệt từng ký tự bằng `for ch in s:`.

In [2]:
s = "1.2.3.4"
print("s[0] =", s[0], "| s[1] =", repr(s[1]), "| len =", len(s))
for i, ch in enumerate(s):      # enumerate cho cả chỉ số lẫn ký tự
    print(i, repr(ch), "là dấu chấm" if ch == "." else "")

s[0] = 1 | s[1] = '.' | len = 7
0 '1' 
1 '.' là dấu chấm
2 '2' 
3 '.' là dấu chấm
4 '3' 
5 '.' là dấu chấm
6 '4' 


`enumerate(s)` sinh từng cặp `(chỉ số, phần tử)`. `repr(x)` in giá trị kèm dấu nháy để thấy rõ đó là chuỗi.

### 2.2. Chuỗi **bất biến** — không sửa tại chỗ được

Không thể viết `s[1] = "[.]"`. Mọi thao tác "sửa" chuỗi thực chất **tạo chuỗi mới**.

In [3]:
s = "1.2"
try:
    s[1] = "[.]"
except TypeError as e:
    print("Lỗi:", e)

t = s + "!"            # tạo chuỗi MỚI
print("s vẫn là:", s, "| t là:", t)

Lỗi: 'str' object does not support item assignment
s vẫn là: 1.2 | t là: 1.2!


`try / except` bắt lỗi để chương trình không dừng — ở đây dùng để **nhìn thấy** thông báo lỗi. Hệ quả của tính bất biến: muốn kết quả khác input, ta phải **xây chuỗi mới**.

### 2.3. `str.replace(old, new)` — thay mọi lần xuất hiện

`s.replace(old, new)` trả về chuỗi mới, trong đó **mọi** lần xuất hiện `old` được thay bằng `new`. Có tham số thứ ba `count` để giới hạn số lần thay.

In [4]:
s = "a.b.c"
print(s.replace(".", "-"))        # a-b-c
print(s.replace(".", "-", 1))     # chỉ thay lần đầu: a-b.c
print(s.replace("x", "y"))        # không có 'x' -> giữ nguyên
print("s gốc:", s)                # replace không đổi s

a-b-c
a-b.c
a.b.c
s gốc: a.b.c


### 2.4. `str.split(sep)` và `sep.join(list)`

- `"1.2.3.4".split(".")` cắt chuỗi tại mỗi `.` → `['1', '2', '3', '4']` (list các mảnh).
- `"-".join(['1','2','3'])` nối các mảnh, chèn `"-"` **giữa** chúng → `"1-2-3"`.

Chú ý cú pháp: `join` là phương thức của **chuỗi ngăn cách**, nhận vào list.

In [5]:
parts = "255.100.50.0".split(".")
print(parts, "| số mảnh:", len(parts))
print("-".join(parts))
print("[.]".join(parts))
print("".join(["a", "b", "c"]))    # nối liền không ngăn cách

['255', '100', '50', '0'] | số mảnh: 4
255-100-50-0
255[.]100[.]50[.]0
abc


## 3. Cách 1 — Duyệt từng ký tự, cộng dồn chuỗi (trực giác)

Người mới thường nghĩ: "đi qua từng ký tự; gặp `.` thì thêm `[.]`, không thì thêm chính ký tự đó".

In [6]:
def defang_concat(address: str) -> str:
    result = ""
    for ch in address:
        if ch == ".":
            result += "[.]"
        else:
            result += ch
    return result

print(defang_concat("1.1.1.1"))
print(defang_concat("255.100.50.0"))

1[.]1[.]1[.]1
255[.]100[.]50[.]0


### 3.1. Nút thắt (về lý thuyết): `+=` trên chuỗi

Vì chuỗi bất biến, `result += ch` về nguyên tắc **tạo chuỗi mới** và sao chép toàn bộ `result` cũ. Nếu chuỗi dài `L`, tổng chi phí có thể là `1 + 2 + ... + L = O(L²)`.

(CPython có tối ưu đặc biệt giúp `+=` thường nhanh, nhưng **không được đảm bảo** ở mọi nơi. Với IP tối đa 15 ký tự thì không đáng kể — nhưng thói quen tốt cho chuỗi dài là dùng list + `join`.)

Thí nghiệm: tạo chuỗi mới hay không — dùng `id()` (định danh đối tượng).

In [7]:
a = "abc"
b2 = a
print("trước:", id(a) == id(b2))
a = a + "d"          # a trỏ tới đối tượng MỚI
print("sau  :", id(a) == id(b2), "| a =", a, "| b2 =", b2)

trước: True
sau  : False | a = abcd | b2 = abc


## 4. Cách 2 — Gom vào list rồi `join` (cách xây chuỗi chuẩn)

List **có thể thay đổi** (mutable): `append` thêm phần tử vào cuối với chi phí trung bình `O(1)`. Cuối cùng `"".join(parts)` nối tất cả trong **một lần**, `O(L)`.

In [8]:
parts = []
parts.append("1")
parts.append("[.]")
parts.append("2")
print(parts, "->", "".join(parts))

['1', '[.]', '2'] -> 1[.]2


In [9]:
from typing import List

def defang_list_join(address: str) -> str:
    out: List[str] = []
    for ch in address:
        out.append("[.]" if ch == "." else ch)   # biểu thức điều kiện một dòng
    return "".join(out)

print(defang_list_join("1.1.1.1"))
print(defang_list_join("255.100.50.0"))

1[.]1[.]1[.]1
255[.]100[.]50[.]0


`A if điều_kiện else B` là **biểu thức điều kiện** (ternary): trả `A` nếu điều kiện đúng, ngược lại `B`. Nó gọn hơn `if/else` 4 dòng khi chỉ chọn giữa 2 giá trị.

In [10]:
for ch in ["7", "."]:
    print(repr(ch), "->", "[.]" if ch == "." else ch)

'7' -> 7
'.' -> [.]


## 5. Insight — bài này chính là "thay thế chuỗi con"

Nhìn lại: ta không cần tự duyệt. Đề chính là định nghĩa của `replace(".", "[.]")`. Hoặc nghĩ theo **cấu trúc** của IP: IP = 4 mảnh số nối bằng `.` → tách bằng `.` rồi nối lại bằng `[.]`.

Ba góc nhìn, cùng một kết quả:
1. **Theo ký tự:** mỗi ký tự `.` → `[.]` (cách 1, 2).
2. **Theo chuỗi con:** thay mọi `"."` (cách 3: `replace`).
3. **Theo cấu trúc:** `split` + `join` (cách 4).
4. **Theo mẫu (pattern):** biểu thức chính quy (cách 5: `re.sub`).

## 6. Cách 3 — `str.replace` (lời giải chính)

In [11]:
def defang_replace(address: str) -> str:
    return address.replace(".", "[.]")

print(defang_replace("1.1.1.1"))
print(defang_replace("255.100.50.0"))

1[.]1[.]1[.]1
255[.]100[.]50[.]0


### 6.1. Câu hỏi hay: `replace` có thay lại dấu `.` **mới** nằm trong `"[.]"` không?

Không. `replace` quét chuỗi **gốc** một lượt từ trái sang phải; phần đã thay thế không bị quét lại. Nên không xảy ra vòng lặp vô hạn kiểu `[[[.]]]`.

In [12]:
s = "1.2"
once = s.replace(".", "[.]")
twice = once.replace(".", "[.]")     # gọi 2 lần mới bị "bọc" thêm
print("1 lần :", once)
print("2 lần :", twice)

1 lần : 1[.]2
2 lần : 1[[.]]2


## 7. Cách 4 — `split` + `join`

In [13]:
def defang_split_join(address: str) -> str:
    return "[.]".join(address.split("."))

print(defang_split_join("1.1.1.1"))
print(defang_split_join("255.100.50.0"))

1[.]1[.]1[.]1
255[.]100[.]50[.]0


## 8. Cách 5 — Biểu thức chính quy `re.sub`

Module `re` (regular expression — biểu thức chính quy) của thư viện chuẩn dùng để tìm/thay theo **mẫu**. Trong mẫu regex, `.` có nghĩa đặc biệt là "**bất kỳ ký tự nào**", nên muốn khớp dấu chấm thật phải **escape**: `\.` (hoặc `[.]`).

`re.sub(pattern, repl, s)` thay mọi đoạn khớp `pattern` bằng `repl`. Tiền tố `r"..."` là **raw string**: Python không xử lý `\` như ký tự thoát, giúp viết regex dễ hơn.

In [14]:
import re

s = "1.1.1.1"
print("mẫu '.'  (SAI, khớp mọi ký tự):", re.sub(".", "[.]", s))
print(r"mẫu '\.' (đúng):", re.sub(r"\.", "[.]", s))

mẫu '.'  (SAI, khớp mọi ký tự): [.][.][.][.][.][.][.]
mẫu '\.' (đúng): 1[.]1[.]1[.]1


In [15]:
def defang_regex(address: str) -> str:
    return re.sub(r"\.", "[.]", address)

print(defang_regex("255.100.50.0"))

255[.]100[.]50[.]0


## 9. TRACE từng bước — cách list + join

Ví dụ `"1.10.2.0"`:

| i | ch | là `.`? | thêm vào out | out sau bước |
|---:|:---:|:---:|---|---|
| 0 | `1` | Không | `1` | `['1']` |
| 1 | `.` | Có | `[.]` | `['1','[.]']` |
| 2 | `1` | Không | `1` | `['1','[.]','1']` |
| 3 | `0` | Không | `0` | `[..., '0']` |
| 4 | `.` | Có | `[.]` | `[..., '[.]']` |
| 5 | `2` | Không | `2` | `[..., '2']` |
| 6 | `.` | Có | `[.]` | `[..., '[.]']` |
| 7 | `0` | Không | `0` | `[..., '0']` |

`"".join(out)` → `"1[.]10[.]2[.]0"`.

In [16]:
def trace_defang(address: str) -> str:
    out = []
    for i, ch in enumerate(address):
        piece = "[.]" if ch == "." else ch
        out.append(piece)
        print(f"i={i}, ch={ch!r}, là '.'? {ch == '.'!s:<5} -> thêm {piece!r:<6} | ghép tạm: {''.join(out)}")
    result = "".join(out)
    print("Kết quả:", result)
    return result

_ = trace_defang("1.10.2.0")

i=0, ch='1', là '.'? False -> thêm '1'    | ghép tạm: 1
i=1, ch='.', là '.'? True  -> thêm '[.]'  | ghép tạm: 1[.]
i=2, ch='1', là '.'? False -> thêm '1'    | ghép tạm: 1[.]1
i=3, ch='0', là '.'? False -> thêm '0'    | ghép tạm: 1[.]10
i=4, ch='.', là '.'? True  -> thêm '[.]'  | ghép tạm: 1[.]10[.]
i=5, ch='2', là '.'? False -> thêm '2'    | ghép tạm: 1[.]10[.]2
i=6, ch='.', là '.'? True  -> thêm '[.]'  | ghép tạm: 1[.]10[.]2[.]
i=7, ch='0', là '.'? False -> thêm '0'    | ghép tạm: 1[.]10[.]2[.]0
Kết quả: 1[.]10[.]2[.]0


Trong f-string, `{x!r}` nghĩa là in `repr(x)` (có dấu nháy), `{x!s}` là `str(x)`, `:<6` căn trái độ rộng 6 ký tự — giúp bảng in thẳng cột.

Trace cách `split` + `join` trên cùng input:

In [17]:
addr = "1.10.2.0"
parts = addr.split(".")
print("Bước 1 — split('.'):", parts)
print("Bước 2 — '[.]'.join(...):", "[.]".join(parts))
print("Số lần chèn '[.]' = số mảnh - 1 =", len(parts) - 1)

Bước 1 — split('.'): ['1', '10', '2', '0']
Bước 2 — '[.]'.join(...): 1[.]10[.]2[.]0
Số lần chèn '[.]' = số mảnh - 1 = 3


## 10. Code nộp LeetCode

Bản chính (`replace`):

In [18]:
class Solution:
    def defangIPaddr(self, address: str) -> str:
        return address.replace(".", "[.]")

sol = Solution()
print(sol.defangIPaddr("1.1.1.1"))
print(sol.defangIPaddr("255.100.50.0"))

1[.]1[.]1[.]1
255[.]100[.]50[.]0


Bản không dùng hàm thư viện xử lý chuỗi (phù hợp khi phỏng vấn yêu cầu "tự viết"):

In [19]:
class SolutionManual:
    def defangIPaddr(self, address: str) -> str:
        out = []
        for ch in address:
            if ch == ".":
                out.append("[.]")
            else:
                out.append(ch)
        return "".join(out)

print(SolutionManual().defangIPaddr("255.100.50.0"))

255[.]100[.]50[.]0


## 11. Vì sao đúng? — và Độ phức tạp

**Tính đúng (cách duyệt):** invariant — sau khi xử lý `k` ký tự đầu, `"".join(out)` bằng đúng phiên bản defang của `address[:k]`. Bước `k+1` chỉ nối thêm ảnh của đúng một ký tự (`.` → `[.]`, ký tự khác giữ nguyên), nên invariant được giữ. Khi `k = len(address)` ta có toàn bộ kết quả. Các ký tự không phải `.` giữ nguyên thứ tự, nên không mất/đảo dữ liệu.

**Tính đúng (`split` + `join`):** `split(".")` cho các mảnh nằm giữa các dấu chấm; `join` chèn `[.]` vào **đúng** những chỗ giữa hai mảnh liền nhau — chính là vị trí các dấu chấm cũ.

Gọi `L = len(address)` (với IPv4 thì `7 <= L <= 15`, nên mọi cách đều coi như `O(1)` thực tế).

| Cách | Time | Space |
|---|---|---|
| 1. `+=` từng ký tự | `O(L)` nhờ tối ưu của CPython; `O(L²)` trong trường hợp xấu | `O(L)` |
| 2. list + `join` | `O(L)` | `O(L)` |
| 3. `replace` | `O(L)` | `O(L)` |
| 4. `split` + `join` | `O(L)` | `O(L)` |
| 5. `re.sub` | `O(L)` (+ chi phí biên dịch regex) | `O(L)` |

Space `O(L)` là cho chuỗi kết quả — bắt buộc vì chuỗi bất biến.

## 12. Bộ test — mẫu, biên, đối chiếu

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS {actual}")

cases = [
    ("mẫu 1", "1.1.1.1", "1[.]1[.]1[.]1"),
    ("mẫu 2", "255.100.50.0", "255[.]100[.]50[.]0"),
    ("toàn số 0", "0.0.0.0", "0[.]0[.]0[.]0"),
    ("dài nhất", "255.255.255.255", "255[.]255[.]255[.]255"),
    ("độ dài khác nhau", "10.0.200.3", "10[.]0[.]200[.]3"),
]
for name, inp, exp in cases:
    check(name, Solution().defangIPaddr(inp), exp)

mẫu 1                  PASS 1[.]1[.]1[.]1
mẫu 2                  PASS 255[.]100[.]50[.]0
toàn số 0              PASS 0[.]0[.]0[.]0
dài nhất               PASS 255[.]255[.]255[.]255
độ dài khác nhau       PASS 10[.]0[.]200[.]3


Đối chiếu 5 cách trên **mọi** IP sinh ngẫu nhiên (có seed), và kiểm tra tính chất: độ dài tăng đúng 6, đếm được đúng 3 lần `"[.]"`, bỏ `[` `]` thì ra lại input.

In [21]:
import random
rng = random.Random(1108)
funcs = [defang_concat, defang_list_join, defang_replace, defang_split_join, defang_regex,
         Solution().defangIPaddr, SolutionManual().defangIPaddr]
for _ in range(3000):
    ip = ".".join(str(rng.randint(0, 255)) for _ in range(4))
    outs = {f(ip) for f in funcs}
    assert len(outs) == 1, (ip, outs)          # tất cả cách cho cùng kết quả
    out = outs.pop()
    assert len(out) == len(ip) + 6
    assert out.count("[.]") == 3
    assert out.replace("[", "").replace("]", "") == ip
print("3000 IP ngẫu nhiên: 7 hàm khớp nhau + đúng tính chất")

3000 IP ngẫu nhiên: 7 hàm khớp nhau + đúng tính chất


Trong cell trên, `{f(ip) for f in funcs}` là **set comprehension**: tạo một `set` (tập hợp, tự loại trùng). Nếu mọi hàm trả cùng một chuỗi thì set chỉ có 1 phần tử.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Gọi `replace` mà không dùng giá trị trả về

In [22]:
def wrong_ignore_return(address):
    address.replace(".", "[.]")      # SAI: kết quả bị bỏ đi
    return address

print("sai :", wrong_ignore_return("1.1.1.1"))
print("đúng:", defang_replace("1.1.1.1"))

sai : 1.1.1.1
đúng: 1[.]1[.]1[.]1


**Vì sao sai:** chuỗi bất biến, `replace` **trả về chuỗi mới** chứ không sửa `address`. **Sửa:** `return address.replace(".", "[.]")` (hoặc gán lại `address = address.replace(...)`).

### Lỗi 2: Regex quên escape dấu chấm

In [23]:
print("sai :", re.sub(".", "[.]", "1.1"))
print("đúng:", re.sub(r"\.", "[.]", "1.1"))

sai : [.][.][.]
đúng: 1[.]1


**Vì sao sai:** trong regex, `.` khớp **mọi** ký tự nên mọi ký tự đều bị thay. **Sửa:** `r"\."` hoặc `re.escape(".")`.

### Lỗi 3: `join` gọi sai phía

In [24]:
parts = ["1", "2", "3"]
try:
    print(parts.join("[.]"))         # SAI: list không có join
except AttributeError as e:
    print("Lỗi:", e)
print("đúng:", "[.]".join(parts))

Lỗi: 'list' object has no attribute 'join'
đúng: 1[.]2[.]3


### Lỗi 4: Sửa chuỗi tại chỗ bằng chỉ số

In [25]:
def wrong_index_assign(address):
    s = address
    for i in range(len(s)):
        if s[i] == ".":
            s[i] = "[.]"             # SAI: TypeError
    return s

try:
    wrong_index_assign("1.1")
except TypeError as e:
    print("Lỗi:", e)

Lỗi: 'str' object does not support item assignment


**Sửa:** xây chuỗi mới (list + join), như cách 2.

### Lỗi 5: Đổi kích thước list trong khi duyệt theo chỉ số

Nếu chuyển chuỗi thành list ký tự rồi **chèn** `[`, `]` vào list trong lúc đang duyệt `range(len(...))` cũ, chỉ số sẽ lệch. Quy tắc: **không sửa cấu trúc đang duyệt**; xây cấu trúc mới.

In [26]:
def wrong_insert_while_iterating(address):
    chars = list(address)
    for i in range(len(chars)):              # len tính 1 lần lúc đầu
        if chars[i] == ".":
            chars.insert(i, "[")             # chèn làm mọi ký tự dời sang phải
            chars.insert(i + 2, "]")
    return "".join(chars)

print("sai :", wrong_insert_while_iterating("1.1.1.1"))
print("đúng:", defang_replace("1.1.1.1"))

sai : 1[[[[[[.]]]]]]1.1.1
đúng: 1[.]1[.]1[.]1


**Vì sao sai:** sau lần chèn đầu, dấu `.` bị dời tới vị trí `i+1`; vòng lặp kế tiếp lại gặp đúng dấu `.` đó và chèn thêm, trong khi `range` vẫn dừng theo độ dài **cũ** nên phần cuối chuỗi không được xét — kết quả bị bọc nhiều lớp ngoặc ở dấu chấm đầu tiên, còn các dấu chấm sau không được xử lý.

## 14. Mẹo nhận dạng pattern

- "Thay mọi X bằng Y" trong chuỗi → `str.replace` là lựa chọn đầu tiên.
- Chuỗi có cấu trúc **phân tách bởi ký tự** (IP, CSV, đường dẫn, ngày `yyyy-mm-dd`) → `split` để xử lý từng mảnh, `join` để ghép lại.
- Xây chuỗi qua nhiều bước → **list + `"".join`**.
- Thay theo **mẫu** phức tạp (chữ số, khoảng trắng nhiều, …) → `re.sub`, nhớ escape ký tự đặc biệt `. * + ? ( ) [ ] { } ^ $ | \`.
- Bài cùng họ: #709 To Lower Case, #1528 Shuffle String, #1678 Goal Parser Interpretation, #468 Validate IP Address, #929 Unique Email Addresses.

## 15. Biến thể

### 15.1. Refang — đảo ngược thao tác

In [27]:
def refang(s: str) -> str:
    return s.replace("[.]", ".")

d = defang_replace("192.168.1.1")
print(d, "->", refang(d))
assert refang(defang_replace("192.168.1.1")) == "192.168.1.1"

192[.]168[.]1[.]1 -> 192.168.1.1


### 15.2. Thay nhiều ký tự cùng lúc — bảng ánh xạ `dict` + `str.translate`

`str.maketrans(dict)` tạo bảng dịch từ ký tự → chuỗi; `s.translate(table)` áp dụng trong một lượt. Ví dụ "defang" cả URL: `.` → `[.]`, `:` → `[:]`.

In [28]:
table = str.maketrans({".": "[.]", ":": "[:]"})
print("http://evil.example.com".translate(table))
print("1.1.1.1".translate(str.maketrans({".": "[.]"})))

http[:]//evil[.]example[.]com
1[.]1[.]1[.]1


### 15.3. Kiểm tra chuỗi có phải IPv4 hợp lệ (ý tưởng của #468)

4 mảnh, mỗi mảnh toàn chữ số, giá trị `0..255`, không có số 0 ở đầu (trừ chính `"0"`).

In [29]:
def is_valid_ipv4(s: str) -> bool:
    parts = s.split(".")
    if len(parts) != 4:
        return False
    for p in parts:
        if not p.isdigit():                 # rỗng hoặc có ký tự lạ
            return False
        if len(p) > 1 and p[0] == "0":      # "01" không hợp lệ
            return False
        if int(p) > 255:
            return False
    return True

for s in ["1.1.1.1", "255.255.255.255", "256.1.1.1", "1.1.1", "01.1.1.1", "1..1.1", "a.b.c.d"]:
    print(f"{s:<16} -> {is_valid_ipv4(s)}")

1.1.1.1          -> True
255.255.255.255  -> True
256.1.1.1        -> False
1.1.1            -> False
01.1.1.1         -> False
1..1.1           -> False
a.b.c.d          -> False


## 16. Bài tập (tăng dần)

**Bài 1 — Dự đoán output:** `"a.b".split(".")`, `".a.".split(".")`, `"-".join("abc")` cho ra gì? (Gợi ý: `join` nhận *bất kỳ* dãy chuỗi nào, kể cả một chuỗi.)

**Bài 2 — Tính tay:** defang `"8.8.4.4"` và cho biết độ dài kết quả.

**Bài 3 — TODO:** viết defang bằng list + join, không dùng `replace`/`split`/`re`.

In [30]:
def exercise_defang(address: str) -> str:
    # TODO 1: out = []
    # TODO 2: duyệt ch; nếu '.', append '[.]', ngược lại append ch
    # TODO 3: return ''.join(out)
    pass

# assert exercise_defang("1.1.1.1") == "1[.]1[.]1[.]1"

**Bài 4 — Refang thủ công:** chuyển `"1[.]2[.]3[.]4"` về `"1.2.3.4"` **không** dùng `replace`. Gợi ý: duyệt bằng chỉ số `i`; nếu `s[i:i+3] == "[.]"` thì thêm `.` và nhảy `i += 3`.

In [31]:
def exercise_refang_manual(s: str) -> str:
    # TODO
    pass

# assert exercise_refang_manual("1[.]2[.]3[.]4") == "1.2.3.4"

**Bài 5 — Defang có điều kiện:** chỉ defang nếu chuỗi là IPv4 hợp lệ; nếu không, trả nguyên chuỗi.

In [32]:
def exercise_safe_defang(s: str) -> str:
    # TODO
    pass

# assert exercise_safe_defang("8.8.8.8") == "8[.]8[.]8[.]8"
# assert exercise_safe_defang("999.1.1.1") == "999.1.1.1"

**Bài 6 — Mini challenge:** viết hàm `replace_all(s, old, new)` tự cài đặt `str.replace` (không gọi `replace`, `split`, `re`), với `old` là chuỗi **không rỗng** có thể dài nhiều ký tự. Phải quét không chồng lấn, từ trái sang phải giống Python.

In [33]:
def exercise_replace_all(s: str, old: str, new: str) -> str:
    # TODO
    pass

# assert exercise_replace_all("1.1.1.1", ".", "[.]") == "1[.]1[.]1[.]1"
# assert exercise_replace_all("aaaa", "aa", "b") == "bb"

### 16.1. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | split giữ cả mảnh rỗng | `".a."` có 2 dấu chấm → 3 mảnh | `['', 'a', '']`; `join("abc")` duyệt từng ký tự |
| 2 | 3 dấu chấm | mỗi `.` thêm 2 ký tự | `7 + 6 = 13` |
| 3 | Làm như mục 4 | biểu thức điều kiện | `out.append("[.]" if ch == "." else ch)` |
| 4 | Dùng `while i < len(s)` | so khớp `s[i:i+3]` | khớp → append `.`, `i += 3`; không → append `s[i]`, `i += 1` |
| 5 | Dùng lại `is_valid_ipv4` | rẽ nhánh 2 trường hợp | `return defang(s) if is_valid_ipv4(s) else s` |
| 6 | Giống bài 4 nhưng tổng quát | so `s[i:i+len(old)] == old` | khớp → append `new`, nhảy `len(old)` |

## 17. Đáp án tham khảo

- **Bài 1:** `['a', 'b']`; `['', 'a', '']`; `"a-b-c"`.
- **Bài 2:** `"8[.]8[.]4[.]4"`, độ dài `13`.

In [34]:
def answer_defang(address):
    out = []
    for ch in address:
        out.append("[.]" if ch == "." else ch)
    return "".join(out)

def answer_refang_manual(s):
    out, i = [], 0
    while i < len(s):
        if s[i:i + 3] == "[.]":
            out.append("."); i += 3
        else:
            out.append(s[i]); i += 1
    return "".join(out)

def answer_safe_defang(s):
    return answer_defang(s) if is_valid_ipv4(s) else s

def answer_replace_all(s, old, new):
    out, i, m = [], 0, len(old)
    while i < len(s):
        if s[i:i + m] == old:
            out.append(new); i += m
        else:
            out.append(s[i]); i += 1
    return "".join(out)

assert "a.b".split(".") == ["a", "b"]
assert ".a.".split(".") == ["", "a", ""]
assert "-".join("abc") == "a-b-c"
assert answer_defang("8.8.4.4") == "8[.]8[.]4[.]4" and len(answer_defang("8.8.4.4")) == 13
assert answer_defang("1.1.1.1") == "1[.]1[.]1[.]1"
assert answer_refang_manual("1[.]2[.]3[.]4") == "1.2.3.4"
assert answer_safe_defang("8.8.8.8") == "8[.]8[.]8[.]8"
assert answer_safe_defang("999.1.1.1") == "999.1.1.1"
assert answer_replace_all("1.1.1.1", ".", "[.]") == "1[.]1[.]1[.]1"
assert answer_replace_all("aaaa", "aa", "b") == "bb"

rng = random.Random(42)
for _ in range(2000):
    s = "".join(rng.choice("ab.") for _ in range(rng.randint(0, 10)))
    old = "".join(rng.choice("ab.") for _ in range(rng.randint(1, 3)))
    assert answer_replace_all(s, old, "X") == s.replace(old, "X"), (s, old)
    assert answer_refang_manual(answer_defang(s)) == s
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 18. Template nhớ nhanh + checklist submit

```python
# Thay thế đơn giản
return address.replace(".", "[.]")

# Theo cấu trúc
return "[.]".join(address.split("."))

# Tự xây chuỗi
out = []
for ch in s:
    out.append(mapping_of(ch))
return "".join(out)
```

**Checklist:**
- [ ] Có `return` giá trị mới (chuỗi bất biến, `replace` không sửa tại chỗ)?
- [ ] Regex có escape `\.`?
- [ ] `join` gọi trên chuỗi ngăn cách: `"[.]".join(parts)`?
- [ ] Không sửa cấu trúc đang duyệt?
- [ ] Kết quả dài hơn input đúng 6 ký tự với IPv4?